<details><summary style="display:list-item; font-size:16px; color:blue;">Jupyter Help</summary>
    
Having trouble testing your work? Double-check that you have followed the steps below to write, run, save, and test your code!

[Click here for a walkthrough GIF of the steps below](https://static-assets.codecademy.com/Courses/ds-python/jupyter-help.gif)

Run all initial cells to import libraries and datasets. Then follow these steps for each question:

1. Add your solution to the cell with `## YOUR SOLUTION HERE ##`.
2. Run the cell by selecting the `Run` button or the `Shift`+`Enter` keys.
3. Save your work by selecting the `Save` button, the `command`+`s` keys (Mac), or `control`+`s` keys (Windows).
4. Select the `Test Work` button at the bottom left to test your work.

![Screenshot of the buttons at the top of a Jupyter Notebook. The Run and Save buttons are highlighted](https://static-assets.codecademy.com/Paths/ds-python/jupyter-buttons.png)

#### Setup Cell

Run the setup cell below to import the libraries required to manage richer state in LangGraph workflows.

`Annotated` pairs a state field with a reducer that controls how it is updated — here, `operator.add` lets `edit_log` accumulate values across nodes instead of replacing them.

The language model `llm` is initialized using `gpt-4o` with a temperature of `0.5`.

**Note:** The OpenAI API key is already configured in this workspace, so no additional setup is needed. When working in an external environment, refer to the [OpenAI API](https://platform.openai.com/docs/quickstart?language=python) documentation for details on configuring your API key.

In [ ]:
# Third-party imports
from langgraph.graph import StateGraph, START, END
from typing_extensions import TypedDict
from typing import Annotated
from langchain_openai import ChatOpenAI
import operator
import warnings

warnings.filterwarnings("ignore")

# Initialize the language model
llm = ChatOpenAI(model="gpt-4o", temperature=0.5)

#### Checkpoint 1/4: Define the State Schema with a Reducer

Let's define a state structure that tracks data throughout the publishing workflow, including a record of edits.

- Create a `PublishingState` class that inherits from `TypedDict` with the following string fields: `topic`, `content_type`, `draft`, and `final_content`.
- Include an `edit_log` field that uses `Annotated` with a list type and the `operator.add` reducer to accumulate editing actions across the workflow.

**Note:** Don't forget to run the cell and save the notebook before selecting `Test Work`. Open the `Jupyter Help` toggle at the top of the notebook for more details.

In [ ]:
# Define the state schema for the pipeline
class PublishingState(TypedDict):
    topic: str
    content_type: str
    draft: str
    final_content: str
    edit_log: Annotated[list, operator.add]

#### Checkpoint 2/4: Create Nodes that Update Multiple Fields

With our state schema in place, let's create two node functions that accept a `state` parameter of type `PublishingState` to handle content creation and formatting.
 
- **Drafting:** Create a function `draft_node` that generates a short post (2–3 paragraphs) based on the `topic` and `content_type`, and returns a dictionary with keys `draft` (the generated content) and `edit_log`.

- **Formatting:** Create a function `format_node` that formats the content for publication based on the `draft` and `content_type`, and returns a dictionary with keys `final_content` (the formatted content) and `edit_log`.

**Note:** Don't forget to run the cell and save the notebook before selecting `Test Work`. Open the `Jupyter Help` toggle at the top of the notebook for more details.

In [ ]:
# Define a node that creates initial draft content
def draft_node(state: PublishingState):
    result = llm.invoke(
        f"Write a short {state['content_type']} post (2-3 paragraphs) on: {state['topic']}"
    )
    return {
        "draft": result.content,
        "edit_log": [f"Draft created as {state['content_type']}"]
    }

# Define a node that formats the draft for final output
def format_node(state: PublishingState):
    result = llm.invoke(
        f"Format this {state['content_type']} for publication. Add appropriate structure "
        f"(title and headings for blog; hashtags for social media; "
        f"subject line for newsletter):\n{state['draft']}"
    )
    return {
        "final_content": result.content,
        "edit_log": [f"Formatted for {state['content_type']}"]
    }

#### Checkpoint 3/4: Build and Compile the Graph

Now that both node functions are ready, let's build the graph by adding them as nodes, defining the execution flow, and compiling it into a runnable pipeline.

- Create a `StateGraph` instance with `PublishingState` as its state type and assign it to `builder`. 

- Add a `draft` node (using `draft_node`) and a `format` node (using `format_node`), then wire the execution flow by adding edges: `START` → `draft` → `format` → `END`.

- Call `compile()` on `builder` and assign the result to `pipeline`.

**Note:** Don't forget to run the cell and save the notebook before selecting `Test Work`. Open the `Jupyter Help` toggle at the top of the notebook for more details.

In [ ]:
# Initialize a StateGraph with the defined state schema
builder = StateGraph(PublishingState)

# Add nodes to the graph
builder.add_node("draft", draft_node)
builder.add_node("format", format_node)

# Connect the nodes using edges
builder.add_edge(START, "draft")
builder.add_edge("draft", "format")
builder.add_edge("format", END)

# Compile the graph into a runnable pipeline
pipeline = builder.compile()

#### Checkpoint 4/4: Run the Pipeline with Different Content Types

With our pipeline compiled, let's test it by generating content in two different formats using the same topic.

- Invoke the `pipeline` with a state dictionary setting `topic` to the `topic` variable, `content_type` to `"blog"`, and `draft`, `final_content`, and `edit_log` to empty values. Store the result in `result_blog`.

- Invoke the `pipeline` again with `content_type` set to `"social_media"` (and the same empty values for the other fields). Store the result in `result_social`.

- Uncomment the `print()` statements to see the edit log and content previews for both formats.

**Note:** Don't forget to run the cell and save the notebook before selecting `Test Work`. Open the `Jupyter Help` toggle at the top of the notebook for more details.

In [ ]:
topic = "The rise of remote work and its impact on team productivity"

# Invoke the pipeline to generate blog-style output
result_blog = pipeline.invoke({
    "topic": topic,
    "content_type": "blog",
    "draft": "", "final_content": "", "edit_log": []
})

print("=== Blog ===")
print(f"Edit Log: {result_blog['edit_log']}")
print(f"Content Preview:\n{result_blog['final_content'][:500]}...")

print(f"\n{'=' * 100}")

# Invoke the pipeline to generate social-media-style output
result_social = pipeline.invoke({
    "topic": topic,
    "content_type": "social_media",
    "draft": "", "final_content": "", "edit_log": []
})

print("\n=== Social Media ===")
print(f"Edit Log: {result_social['edit_log']}")
print(f"Content Preview:\n{result_social['final_content'][:500]}...")

<details> 

<summary><strong>Summary</strong></summary> 

In this exercise, we explored how state updates work in LangGraph. By default, values are overwritten, but using `Annotated` with a reducer like `operator.add` enables values to accumulate across nodes. 

</details>